# PneumoScan — Explainable Pneumonia Screening on Adult Chest X-rays

**Dataset:** RSNA Pneumonia Detection Challenge (adult, radiologist-annotated boxes)
**Model:** DenseNet121 at **320×320**, mixed precision
**Explainability:** Grad-CAM scored against the boxes, not just displayed

> **Before anything else**
> 1. `Runtime → Change runtime type → **T4 GPU**`
> 2. Accept the competition rules once: https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules
> 3. **Keep this tab open and in the foreground.** Free Colab disconnects on idle,
>    and that is what killed the last run.

**About 1 hour 40 minutes**, most of it unattended.

### Built to survive a disconnect
Checkpoints and logs are written to **Google Drive every epoch**, so if the
session dies you lose the current epoch and nothing else. Section 8 has a resume
cell that continues from where it stopped.

---
1. GPU  2. Code  3. Self-test  4. Drive + config  5. Kaggle
6. Download + CLAHE  7. Cohort  8. Train  9. Results  10. Grad-CAM  11. TFLite

## 1. GPU and dependencies

In [ ]:
!nvidia-smi -L
!pip install -q pydicom kaggle
import tensorflow as tf, keras, pydicom
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| pydicom", pydicom.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "NONE  -> Runtime > Change runtime type > T4 GPU, then rerun")
assert gpus, "no GPU - fix the runtime before continuing"

## 2. Get the project code

Use **2a** if the project is on GitHub, otherwise **2b** to upload `pneumoscan.zip`.
Both are safe to re-run: they replace code only and leave `data/` alone.

In [ ]:
# --- 2a. Clone from GitHub -------------------------------------------------
REPO_URL = ""   # e.g. "https://github.com/Murali1801/pneumoscan.git"

import os, shutil
if REPO_URL:
    shutil.rmtree("/content/project", ignore_errors=True)
    !git clone -q $REPO_URL /content/project
    os.chdir("/content/project")
    print("cloned into", os.getcwd())
    !ls
else:
    print("REPO_URL is empty - use cell 2b instead.")

In [ ]:
# --- 2b. Upload pneumoscan.zip ---------------------------------------------
# Replaces the CODE only. data/, checkpoints/ and reports/ are preserved, so
# this is safe to re-run mid-session to pick up a fix.
import os, glob, zipfile, shutil
from google.colab import files

PROJECT = "/content/project"
KEEP = {"data", "checkpoints", "reports"}

up = files.upload()
name = next(iter(up))
shutil.rmtree("/content/_unzip", ignore_errors=True)
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/_unzip")

root = next(iter(glob.glob("/content/_unzip/**/src/train.py", recursive=True)), None)
assert root, "src/train.py not found inside the zip"
src_root = os.path.dirname(os.path.dirname(root))

for item in os.listdir(PROJECT):
    if item in KEEP:
        continue
    p = os.path.join(PROJECT, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

for item in os.listdir(src_root):
    shutil.move(os.path.join(src_root, item), PROJECT)

os.chdir(PROJECT)
print("project at", os.getcwd(), "| preserved:", [d for d in KEEP if os.path.isdir(d)])
!ls

## 3. Self-test *(~3 min — do not skip)*

Generates a synthetic RSNA-shaped dataset and runs every stage against it:
DICOM decoding, CLAHE, splits, box rescaling, training, **resume**, Grad-CAM,
the localisation metric and the TFLite export.

`SELF-TEST PASSED` means the code and this runtime work. Three minutes here has
already caught more bugs than any other check in this project.

In [ ]:
!python scripts/selftest.py

## 4. Mount Drive and set the run configuration

Everything the run produces goes to Drive as it happens, so a dropped session
costs at most one epoch.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
IMG_SIZE = 320
TAG      = "320"
DRIVE    = "/content/drive/MyDrive/pneumoscan"
CKPT     = f"{DRIVE}/checkpoints"
REPORTS  = f"{DRIVE}/reports"
SPLITS   = "data/splits_320.csv"
PROC     = "data/processed_320"
RUN      = f"densenet121_{TAG}"

PREP  = f"--img-size {IMG_SIZE} --processed-dir {PROC} --splits-csv {SPLITS} --out-dir {REPORTS}"
TRAIN = f"--img-size {IMG_SIZE} --splits-csv {SPLITS} --out-dir {REPORTS} --ckpt-dir {CKPT} --tag {TAG}"

import os
os.makedirs(CKPT, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)
print("run     :", RUN)
print("outputs :", DRIVE)

## 5. Kaggle credentials

Kaggle → avatar → **Settings** → **API** → *Create New Token*.

Preferred: add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab **Secrets** (🔑 in the
sidebar) with notebook access on. Otherwise upload `kaggle.json` when prompted.

In [ ]:
import os, json, pathlib

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("using Colab secrets for", os.environ["KAGGLE_USERNAME"])
except Exception as e:
    print("secrets unavailable (%s) - upload kaggle.json instead" % type(e).__name__)
    from google.colab import files
    up = files.upload()
    creds = json.loads(next(iter(up.values())))
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]

d = pathlib.Path.home() / ".kaggle"; d.mkdir(exist_ok=True)
(d / "kaggle.json").write_text(json.dumps({
    "username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
(d / "kaggle.json").chmod(0o600)
print("kaggle.json ready")

## 6. Download, CLAHE, split  *(~20 min)*

Processed images stay on local disk, not Drive — 26,684 small PNGs read from
Drive every epoch would be far slower than re-creating them.

**403 here means the competition rules are not accepted yet.**

In [ ]:
!python scripts/prepare_data.py --download {PREP}

In [ ]:
import pandas as pd
from IPython.display import Image, display

print("Split sizes");  display(pd.read_csv(f"{REPORTS}/dataset_summary.csv"))
print("RSNA classes"); display(pd.read_csv(f"{REPORTS}/detailed_class_summary.csv", index_col=0))
display(Image(f"{REPORTS}/clahe_examples.png", width=760))

### Check these numbers

Across all splits: **26,684** images, **6,012** Lung Opacity, 8,851 Normal,
11,821 No Lung Opacity / Not Normal, and **9,555** boxes. If those match, the
ingest is correct.

## 7. Cohort table — the evidence the data is adult

Reads age, sex and view straight from the DICOM headers. `pct_under_18` should
be near zero. This is what lets the report *show* the cohort rather than assert it.

In [ ]:
!python scripts/cohort.py --splits-csv {SPLITS} --out-dir {REPORTS}

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(f"{REPORTS}/cohort.csv"))
display(Image(f"{REPORTS}/age_histogram.png", width=680))

## 8. Train  *(~75 min)*

320×320 with mixed precision — float16 compute on the T4's tensor cores, float32
master weights. Roughly 1.5× faster than float32, and the checkpoint is rebuilt
in plain float32 at the end so Grad-CAM and TFLite see an ordinary graph.

Two phases: 3 warm-up epochs with the backbone frozen, then fine-tuning that
early-stops on validation AUROC.

**Your 224px baseline was test AUROC 0.876, validation 0.887.** Watch `val_auc`
pass 0.887 during fine-tuning.

In [ ]:
!python src/train.py {TRAIN} --batch-size 24 --mixed-precision --finetune-epochs 16

### If the session dropped — resume here

Re-run sections 1, 2, 4, 5, then **section 6** (the download is needed again, but
already-processed images are skipped), then this cell instead of the one above.
It reads the epoch it stopped at from the Drive logs and continues.

In [ ]:
!python src/train.py {TRAIN} --batch-size 24 --mixed-precision --finetune-epochs 16 --resume

## 9. Results

In [ ]:
import json, pandas as pd
from pathlib import Path

m = json.loads(Path(f"{REPORTS}/{RUN}/metrics.json").read_text())
lo, hi = m["test_auroc_95ci"]
t = m["test"]
prev = (t["confusion_matrix"]["tp"] + t["confusion_matrix"]["fn"]) / sum(t["confusion_matrix"].values())

print(f"{RUN}   ({m['train_minutes']:.0f} min)")
print(f"  AUROC        {t['auroc']:.4f}   (95% CI {lo:.4f}-{hi:.4f})   [224px baseline: 0.8765]")
print(f"  AUPRC        {t['auprc']:.4f}   (prevalence baseline {prev:.4f})")
print(f"  sensitivity  {t['sensitivity_recall']:.4f}")
print(f"  specificity  {t['specificity']:.4f}")
print(f"  NPV          {t['npv']:.4f}")
print(f"  PPV          {t['precision_ppv']:.4f}")
print(f"  accuracy     {t['accuracy']:.4f}   (majority-class baseline {1 - prev:.4f})")
print("\n  Lead with AUROC. Accuracy barely beats the majority baseline at this prevalence.")

In [ ]:
from IPython.display import Image, display
for f in ("training_curves.png", "test_curves.png", "test_confusion.png"):
    display(Image(f"{REPORTS}/{RUN}/{f}", width=820))

## 10. Grad-CAM and the localisation metric

At 320px the feature map is 10×10 rather than 7×7, so the heatmap is finer than
in the 224 run. **Your 224px baseline: pointing game 0.435 at 3.3× chance.**

In [ ]:
!python scripts/make_gradcam_figures.py {TRAIN}

In [ ]:
import pandas as pd
from IPython.display import Image, display

loc = pd.read_csv(f"{REPORTS}/{RUN}/localization.csv")
display(loc[["explainer", "n", "pointing_game", "pointing_game_chance",
             "pointing_game_lift", "energy_pointing_game"]])

for f in ("gradcam_pneumonia.png", "gradcam_errors.png", "gradcam_vs_pp.png"):
    try:
        display(Image(f"{REPORTS}/{RUN}/{f}", width=900))
    except Exception:
        print("missing:", f)

## 11. Export to TensorFlow Lite

Refuses to write the float model if it drifts from Keras by more than 1e-3 on
real test images. Note this model expects **320×320** input, not 224.

In [ ]:
!python src/export_tflite.py {TRAIN}

## 12. Everything is already on Drive

`reports/` and `checkpoints/` were written straight to
`MyDrive/pneumoscan/` throughout, so there is nothing to download. This cell just
zips a copy if you want one locally.

In [ ]:
!zip -qr /content/pneumoscan_320.zip "{DRIVE}" -x "*.npz"
from google.colab import files
files.download("/content/pneumoscan_320.zip")